In [ ]:
!pip install transformers torch pillow pandas numpy scikit-learn jiwer albumentations -q
print("✅ Dependencies installed")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
print(os.listdir("/content/drive/MyDrive/medocr/"))

In [ ]:
import os, shutil, pandas as pd

os.makedirs("/content/data/synthetic/images", exist_ok=True)
os.makedirs("/content/data/iam", exist_ok=True)

# Copy small files locally
shutil.copy(
    "/content/drive/MyDrive/medocr/labels_final.csv",
    "/content/data/labels_final.csv"
)
shutil.copytree(
    "/content/drive/MyDrive/medocr/synthetic",
    "/content/data/synthetic",
    dirs_exist_ok=True
)

# Point IAM directly to Drive
IAM_ROOT = "/content/drive/MyDrive/medocr/iam_words"
print("✅ labels_final.csv copied")
print("✅ synthetic folder copied")
print(f"✅ IAM reading directly from Drive: {IAM_ROOT}")

In [ ]:
df = pd.read_csv("/content/data/labels_final.csv")

def fix_path(path):
    if pd.isna(path):
        return path
    path = str(path).replace("\\", "/")
    if "iam_words/words" in path:
        parts = path.split("iam_words/words/")
        if len(parts) > 1:
            return f"/content/drive/MyDrive/medocr/iam_words/words/{parts[1]}"
    if "synthetic" in path:
        filename = path.split("/")[-1]
        return f"/content/data/synthetic/images/{filename}"
    return path

df["image_path"] = df["image_path"].apply(fix_path)

# Verify paths
found, missing = 0, 0
for path in df["image_path"].sample(50):
    if os.path.exists(path): found += 1
    else: missing += 1

print(f"Path check: {found} found, {missing} missing")
df.to_csv("/content/data/labels_final_colab.csv", index=False)
print("✅ Fixed paths saved")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from transformers import TrOCRProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

df       = pd.read_csv("/content/data/labels_final_colab.csv")
df_train = df[df["split"] == "train"].reset_index(drop=True)
df_val   = df[df["split"] == "val"].reset_index(drop=True)
df_test  = df[df["split"] == "test"].reset_index(drop=True)

print(f"Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}")

class HandwritingDataset(Dataset):
    def __init__(self, df, processor, max_target_length=32):
        self.df               = df
        self.processor        = processor
        self.max_target_length = max_target_length

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:
            image = Image.open(row["image_path"]).convert("RGB")
        except:
            image = Image.new("RGB", (128, 32), color=(255, 255, 255))

        pixel_values = self.processor(
            image, return_tensors="pt"
        ).pixel_values.squeeze()

        labels = self.processor.tokenizer(
            str(row["transcription"]),
            padding="max_length",
            max_length=self.max_target_length,
            truncation=True,
            return_tensors="pt"
        ).input_ids.squeeze()

        labels[labels == self.processor.tokenizer.pad_token_id] = -100
        return {"pixel_values": pixel_values, "labels": labels}

print("✅ Dataset class ready")

In [ ]:
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

print("Loading TrOCR model...")
processor = TrOCRProcessor.from_pretrained("microsoft/trocr-base-handwritten")
model     = VisionEncoderDecoderModel.from_pretrained("microsoft/trocr-base-handwritten")
model     = model.to(device)

model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id           = processor.tokenizer.pad_token_id
model.config.vocab_size             = model.config.decoder.vocab_size

print("✅ Model loaded!")
print(f"   Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
import torch, gc, os

torch.cuda.empty_cache()
gc.collect()
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

BATCH_SIZE = 8

train_dataset = HandwritingDataset(df_train, processor)
val_dataset   = HandwritingDataset(df_val,   processor)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches:   {len(val_loader)}")
print(f"Batch size:    {BATCH_SIZE}")
print("✅ Dataloaders ready")

In [ ]:
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
from tqdm import tqdm

EPOCHS             = 3
LR                 = 5e-5
ACCUMULATION_STEPS = 4
SAVE_DIR           = "/content/drive/MyDrive/medocr/models/trocr-medical"

os.makedirs(SAVE_DIR, exist_ok=True)

optimizer = AdamW(model.parameters(), lr=LR)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=100,
    num_training_steps=len(train_loader) * EPOCHS
)

best_val_loss = float("inf")

for epoch in range(EPOCHS):
    # ── Training ──
    model.train()
    train_loss = 0
    optimizer.zero_grad()
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Train]")

    for i, batch in enumerate(loop):
        pixel_values = batch["pixel_values"].to(device)
        labels       = batch["labels"].to(device)

        outputs = model(pixel_values=pixel_values, labels=labels)
        loss    = outputs.loss / ACCUMULATION_STEPS

        loss.backward()

        if (i + 1) % ACCUMULATION_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        train_loss += loss.item() * ACCUMULATION_STEPS
        loop.set_postfix(loss=loss.item() * ACCUMULATION_STEPS)

        del pixel_values, labels, outputs, loss
        torch.cuda.empty_cache()

    avg_train_loss = train_loss / len(train_loader)

    # ── Validation ──
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Epoch {epoch+1}/{EPOCHS} [Val]"):
            pixel_values = batch["pixel_values"].to(device)
            labels       = batch["labels"].to(device)
            outputs      = model(pixel_values=pixel_values, labels=labels)
            val_loss    += outputs.loss.item()

            del pixel_values, labels, outputs
            torch.cuda.empty_cache()

    avg_val_loss = val_loss / len(val_loader)
    print(f"\nEpoch {epoch+1} — Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        model.save_pretrained(SAVE_DIR)
        processor.save_pretrained(SAVE_DIR)
        print(f"✅ Best model saved to Drive")

print("\n🎉 Training complete!")

In [ ]:
from jiwer import cer, wer

# Copy model locally to avoid HFValidationError
LOCAL_DIR = "/content/model"
if not os.path.exists(LOCAL_DIR):
    shutil.copytree(SAVE_DIR, LOCAL_DIR)

processor_eval = TrOCRProcessor.from_pretrained(LOCAL_DIR)
model_eval     = VisionEncoderDecoderModel.from_pretrained(LOCAL_DIR)
model_eval.eval()

def predict(image_path):
    try:
        image        = Image.open(image_path).convert("RGB")
        pixel_values = processor_eval(image, return_tensors="pt").pixel_values
        with torch.no_grad():
            generated_ids = model_eval.generate(pixel_values)
        return processor_eval.batch_decode(generated_ids, skip_special_tokens=True)[0]
    except:
        return ""

# Fix paths for test set
df_test_eval = df_test.copy()
df_test_eval["image_path"] = df_test_eval["image_path"].apply(fix_path)

print("Running evaluation on 200 test samples...\n")
sample      = df_test_eval.sample(200, random_state=42)
actuals     = []
predictions = []
correct     = 0

for i, (_, row) in enumerate(sample.iterrows()):
    actual    = str(row["transcription"]).strip()
    predicted = predict(row["image_path"]).strip()
    actuals.append(actual)
    predictions.append(predicted)
    if predicted.lower() == actual.lower():
        correct += 1
    if (i + 1) % 50 == 0:
        print(f"  {i+1}/200 done...")

total_cer = cer(actuals, predictions)
total_wer = wer(actuals, predictions)
accuracy  = correct / len(sample) * 100

print(f"\n{'='*45}")
print(f"  EVALUATION RESULTS (200 test samples)")
print(f"{'='*45}")
print(f"  Exact match accuracy : {accuracy:.1f}%")
print(f"  Character Error Rate : {total_cer*100:.2f}%")
print(f"  Word Error Rate      : {total_wer*100:.2f}%")
print(f"{'='*45}")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(4, 3, figsize=(18, 12))
samples   = list(zip(actuals[:12], predictions[:12],
                     sample["image_path"].tolist()[:12]))

for ax, (actual, predicted, path) in zip(axes.flat, samples):
    try:
        img = Image.open(path).convert("RGB")
        ax.imshow(img, cmap="gray")
    except:
        ax.set_facecolor("#f0f0f0")

    match = predicted.lower() == actual.lower()
    color = "green" if match else "red"
    ax.set_title(
        f"Actual:    {actual}\nPredicted: {predicted}",
        fontsize=9, color=color, loc="left"
    )
    ax.axis("off")

plt.suptitle("Model Predictions — Green=Correct, Red=Wrong", fontsize=13)
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/medocr/evaluation_results.png", dpi=150)
plt.show()
print("✅ Saved to Drive")